# Ligand Recombination 

This notebook contains the code to generate the enumeration sets for the CustomKinFragLib validation (see the KinFragLib repository). We create two enumeration sets, one with fragments from CustomKinFragLib and one with fragments rejected by CustomKinFragLib. 

**Note**: Both fragment libraries are available on Zenodo, and should be copied here in the repository to reproduce the results of this notebook. 

## Imports and paths

In [ ]:
from pathlib import Path
import pickle
import sys

from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole
from rdkit.Chem.PropertyMol import PropertyMol
Chem.SetDefaultPickleProperties(Chem.PropertyPickleOptions.AllProps)
import kinase_focused_fragment_library as kffl
import subprocess


In [104]:
HERE = Path(_dh[-1])

project_path = HERE / ".." / ".." / ".." 
sys.path.append(str(project_path / 'KinaseFocusedFragmentLibrary' / 'kinase_focused_fragment_library' / 'ligand_analysis'))
from utils import construct_ligand, read_fragment_library, standardize_mol

In [16]:
sys.path.append(str(project_path / 'KinaseFocusedFragmentLibrary' / 'kinase_focused_fragment_library' / 'recombination'))
from pickle_loader import pickle_loader

In [ ]:
custom_path = HERE / ".." / ".." / ".." / "custom_fragment_library_sampled"
reference_path = HERE / ".." / ".." / ".." / "reference_fragment_library_sampled"
SUBPOCKETS = ['AP', 'FP', 'SE', 'GA', 'B1', 'B2']

## Read fragment libraries

Read the CustomKinFragLib subsampled fragments and the sampled fragments from only rejected fragments in KinFragLib. 

In [90]:
custom_sampled = read_fragment_library(custom_path, SUBPOCKETS)

In [91]:
reference_sampled = read_fragment_library(reference_path, SUBPOCKETS)

In [25]:
def bond_to_str(bonds):
    bond_str = ""
    for b in bonds:
        bond_str += ",".join(b)
        bond_str += "|"
    return bond_str[:-1]

In [105]:
def ligand_recombination(meta, fragment_library):
    # construct ligand, taken from kffl
    try:
        ligand = construct_ligand(meta, fragment_library)
    except Exception as e:
        print(
            f"Error {e}: Molecule construction failed for molecule with fragment IDs {meta.frag_ids}"
        )
        return None

    # standardize molecule
    try:
        ligand = standardize_mol(ligand)
    except Exception as e:
        print(
            f"Error {e}: Molecule standardization failed for molecule with fragment IDs {meta.frag_ids}"
        )
        return None
    ligand.SetProp("frag_ids", ",".join(meta.frag_ids))
    ligand.SetProp("bond_ids", bond_to_str(meta.bonds))
    return ligand

In [ ]:
def combinatory_library(path_combinatorial_library, fragment_library):

    results = []

    # get all paths to pickle files with molecules from recombination
    paths_pickle_combinatorial_library = list(
        (path_combinatorial_library / "results").glob("*.pickle")
    )

    # iterate over pickle files
    for i, path_pickle_combinatorial_library in enumerate(
        paths_pickle_combinatorial_library
    ):
        with open(str(path_pickle_combinatorial_library), "rb") as pickle_in:
            # process ligands in pickle file (returns list of dict)
            results_tmp = [
                ligand_recombination(meta, fragment_library)
                for meta in pickle_loader(pickle_in)
            ]

            # extend results list with ligands from current iteration
            results.extend(results_tmp)
    results = [r for r in results if r is not None]
    print(f"Number of recombined ligands from all iterations: {len(results)}")

    with Chem.SDWriter(
        str(path_combinatorial_library / "combinatorial_library.sdf")
    ) as f:
        for m in results:
            if m is not None:
                f.write(m)

## Recombine molecules
We enumerate all fragment recombinations while respecting the subpocket connectivity and BRICS compatibility. While the KFFL code does not record any partial solutions, we adapted the code to generate all enumerations. 

**Note**: Most parts of the code in this notebook has been taken from KFFL-recombine. 

### Enumeration set with rejected fragments 

In [ ]:
for n_frags in range(2,5): 
    print(f"Maximum number of fragments per ligand: {n_frags}")
    correct = subprocess.run(['kffl-recombination', '-f', str(reference_path), '-o', str(reference_path) + "/combinatorial_library_" + str(n_frags), "-d", str(n_frags), "-s", "AP"],
        check=True, text=True)
    combinatory_library(reference_path / str("combinatorial_library_" + str(n_frags)), reference_sampled)

Maximum number of fragments per ligand: 2
Number of fragments:  382
Write 6821 results to file 0
Number of resulting ligands:  6821
Number of ligands including duplicates:  6992
Overall number of fragments in queue:  83
Time:  0.6582050323486328
Maximum number of fragments per ligand: 3
Number of fragments:  382
Write 168859 results to file 0
Number of resulting ligands:  168859
Number of ligands including duplicates:  329179
Overall number of fragments in queue:  5327
Time:  3.2207672595977783
Maximum number of fragments per ligand: 4
Number of fragments:  382
Write 500000 results to file 0
Write 397833 results to file 1
Number of resulting ligands:  897833
Number of ligands including duplicates:  2261217
Overall number of fragments in queue:  54120
Time:  29.392444849014282


In [ ]:
# merge combinatorial libraries from runs above (with different number of max. fragments per ligand)
with Chem.SDWriter(str(reference_path / "combinatorial_library.sdf")) as w:
    for num_frags in range(2, 5):
        file_name = "combinatorial_library_" + str(num_frags) + "/combinatorial_library.sdf"
        with Chem.SDMolSupplier(str(reference_path / file_name), removeHs=False) as mol_file:
            for m in mol_file:
                if m is not None: 
                    w.write(m)

### Enumeration set from CustomKinFragLib fragments

In [95]:
for n_frags in range(2,5): 
    print(f"Maximum number of fragments per ligand: {n_frags}")
    correct = subprocess.run(['kffl-recombination', '-f', str(custom_path), '-o', str(custom_path) + "/combinatorial_library_" + str(n_frags), "-d", str(n_frags), "-s", "AP"],
        check=True, text=True)
    #combinatory_library(custom_path / str("combinatorial_library_" + str(n_frags)), custom_sampled)

Maximum number of fragments per ligand: 2
Number of fragments:  364
Write 7018 results to file 0
Number of resulting ligands:  7018
Number of ligands including duplicates:  7018
Overall number of fragments in queue:  80
Time:  1.249722957611084
Maximum number of fragments per ligand: 3
Number of fragments:  364
Write 161941 results to file 0
Number of resulting ligands:  161941
Number of ligands including duplicates:  301541
Overall number of fragments in queue:  5184
Time:  3.0277791023254395
Maximum number of fragments per ligand: 4
Number of fragments:  364
Write 500000 results to file 0
Write 266360 results to file 1
Number of resulting ligands:  766360
Number of ligands including duplicates:  1460618
Overall number of fragments in queue:  49958
Time:  20.40142297744751


In [ ]:
with Chem.SDWriter(str(custom_path / "combinatorial_library.sdf")) as w:
    for num_frags in range(2, 5):
        file_name = "combinatorial_library_" + str(num_frags) + "/combinatorial_library.sdf"
        with Chem.SDMolSupplier(str(custom_path / file_name), removeHs=False) as mol_file:
            for m in mol_file:
                if m is not None: 
                    w.write(m)

For the analysis of the enumeration space, we refer to CustomKinFragLib Notebook 2.4 in the KinFragLib GitHub repository. 